# Bedrock Claude 4를 활용한 통합 문서 처리 시스템
(이미지와 PDF 파일 모두 지원)

## 1. 환경 셋업

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install Pillow openpyxl pandas boto3

In [ ]:
import sys
import boto3
import json
import os
from botocore.config import Config
from datetime import datetime

# utils 모듈 임포트
from utils import (
    get_file_type,
    process_document_with_claude,
    format_translation_document,
    save_translation_document,
    create_translation_workflow,
    check_file_paths,
    list_files_in_directory,
    add_python_path
)

# Python 경로 설정
module_path = ".."
add_python_path(module_path)

In [ ]:
# AWS Bedrock 클라이언트 설정

region = "us-west-2"
claude4_model_id = "us.anthropic.claude-sonnet-4-20250514-v1:0"

config = Config(
    read_timeout=300,  # PDF와 이미지 처리를 위해 5분으로 설정
)

client = boto3.client(service_name="bedrock-runtime", region_name=region, config=config)

## 2. 샘플 파일 확인

In [ ]:
# 현재 작업 디렉토리 확인
current_folder = os.getcwd()
print(f"현재 작업 디렉토리: {current_folder}")

# samples 폴더의 파일들 확인
sample_dir = "samples"
if os.path.exists(sample_dir):
    print(f"\n{sample_dir} 폴더의 파일들:")
    
    # 이미지 파일들
    image_files = list_files_in_directory(sample_dir, ['.jpg', '.jpeg', '.png', '.gif', '.bmp'])
    print(f"이미지 파일들: {image_files}")
    
    # PDF 파일들
    pdf_files = list_files_in_directory(sample_dir, ['.pdf'])
    print(f"PDF 파일들: {pdf_files}")
    
    # 전체 파일 목록
    all_files = list_files_in_directory(sample_dir)
    print(f"전체 파일들: {all_files}")
else:
    print(f"'{sample_dir}' 폴더가 존재하지 않습니다.")

## 3. 문서 처리 함수

In [ ]:
def process_document_for_translation(file_path: str, client=None):
    """문서(이미지 또는 PDF)를 처리하여 번역용 엑셀 파일 생성"""
    
    print(f"=== 문서 처리 시작: {file_path} ===")
    
    try:
        # 1. 파일 존재 여부 및 타입 확인
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}")
        
        file_type = get_file_type(file_path)
        print(f"파일 타입: {file_type}")
        
        if file_type == 'unknown':
            raise ValueError(f"지원하지 않는 파일 형식입니다: {file_path}")
        
        # 2. Claude로 텍스트 추출
        print("\n2. Claude 4로 텍스트 추출 중...")
        extracted_text = process_document_with_claude(file_path, client, claude4_model_id)
        print(f"   추출된 텍스트 길이: {len(extracted_text)} 문자")
        
        # 3. 텍스트를 그룹으로 분류 (간단한 방식)
        print("\n3. 텍스트 그룹 분류 중...")
        # 줄바꿈을 기준으로 텍스트를 나누고 빈 줄로 그룹 분리
        lines = extracted_text.split('\n')
        groups = []
        current_group = []
        
        for line in lines:
            line = line.strip()
            if line:  # 빈 줄이 아닌 경우
                current_group.append(line)
            else:  # 빈 줄인 경우 그룹 분리
                if current_group:
                    groups.append(current_group)
                    current_group = []
        
        # 마지막 그룹 추가
        if current_group:
            groups.append(current_group)
        
        # 그룹이 없으면 전체 텍스트를 하나의 그룹으로
        if not groups:
            groups = [[extracted_text]]
        
        print(f"   총 {len(groups)}개의 텍스트 그룹 생성")
        
        # 4. 번역 문서 생성
        print("\n4. 번역 문서 생성 중...")
        document_name = os.path.splitext(os.path.basename(file_path))[0]
        final_file_path = create_translation_workflow(
            grouped_texts=groups,
            document_name=document_name,
            source_lang="Korean",
            target_lang="English"
        )
        
        print(f"\n=== 처리 완료 ===")
        print(f"최종 파일: {final_file_path}")
        
        # 5. 추출된 텍스트를 파일로 저장 (선택사항)
        output_dir = "extracted_texts"
        os.makedirs(output_dir, exist_ok=True)
        text_file_path = os.path.join(output_dir, f"{document_name}_extracted.txt")
        
        with open(text_file_path, "w", encoding="utf-8") as f:
            f.write(extracted_text)
        
        print(f"추출된 텍스트 저장: {text_file_path}")
        
        return final_file_path, groups, extracted_text
        
    except Exception as e:
        print(f"오류 발생: {str(e)}")
        return None, None, None

## 4. 문서 처리 실행

In [ ]:
# 처리할 파일 선택 (이미지 또는 PDF)
# 예시 파일들 - 실제 존재하는 파일로 변경하세요

# 이미지 파일 예시
# target_file = "samples/sample1.jpg"
# target_file = "samples/6.png"

# PDF 파일 예시
target_file = "samples/sample3.pdf"
# target_file = "samples/preregister_web_source.pdf"

# 파일 존재 여부 확인
if os.path.exists(target_file):
    file_type = get_file_type(target_file)
    print(f"선택된 파일: {target_file}")
    print(f"파일 타입: {file_type}")
    print(f"파일 크기: {os.path.getsize(target_file) / (1024*1024):.2f} MB")
else:
    print(f"파일이 존재하지 않습니다: {target_file}")
    print("\n사용 가능한 파일들:")
    sample_files = list_files_in_directory("samples")
    for f in sample_files:
        print(f"  - {f}")

In [ ]:
# 문서 처리 실행
if os.path.exists(target_file):
    print("\n" + "="*60)
    print("문서 처리를 시작합니다...")
    print("="*60)
    
    final_file, groups, extracted_text = process_document_for_translation(target_file, client)
    
    if final_file:
        print(f"\n✅ 성공적으로 완료되었습니다!")
        print(f"📁 번역 문서 위치: {final_file}")
        print(f"📊 텍스트 그룹 수: {len(groups) if groups else 0}")
        
        # 추출된 텍스트 미리보기
        if extracted_text:
            print(f"\n📝 추출된 텍스트 미리보기 (처음 500자):")
            print("-" * 50)
            print(extracted_text[:500] + ("..." if len(extracted_text) > 500 else ""))
            print("-" * 50)
    else:
        print("❌ 처리 중 오류가 발생했습니다.")
else:
    print("❌ 파일이 존재하지 않아 처리할 수 없습니다.")

## 5. 배치 처리 (선택사항)

In [ ]:
# 여러 파일을 한 번에 처리하는 배치 처리 함수
def batch_process_documents(file_list: list, client=None):
    """여러 문서를 배치로 처리"""
    
    results = []
    
    for i, file_path in enumerate(file_list, 1):
        print(f"\n{'='*60}")
        print(f"배치 처리 {i}/{len(file_list)}: {file_path}")
        print(f"{'='*60}")
        
        if not os.path.exists(file_path):
            print(f"❌ 파일이 존재하지 않습니다: {file_path}")
            results.append((file_path, None, "파일 없음"))
            continue
        
        try:
            final_file, groups, extracted_text = process_document_for_translation(file_path, client)
            
            if final_file:
                results.append((file_path, final_file, "성공"))
                print(f"✅ 완료: {final_file}")
            else:
                results.append((file_path, None, "처리 실패"))
                print(f"❌ 실패: {file_path}")
                
        except Exception as e:
            results.append((file_path, None, f"오류: {str(e)}"))
            print(f"❌ 오류: {str(e)}")
    
    # 결과 요약
    print(f"\n{'='*60}")
    print("배치 처리 결과 요약")
    print(f"{'='*60}")
    
    for file_path, result_file, status in results:
        print(f"{os.path.basename(file_path)}: {status}")
    
    return results

# 배치 처리 실행 예시 (필요시 주석 해제)
# batch_files = [
#     "samples/sample1.jpg",
#     "samples/sample3.pdf",
#     "samples/6.png"
# ]
# 
# batch_results = batch_process_documents(batch_files, client)